In [1]:
#!/usr/bin/env python3
"""
External Validation Script - TCGA to PAIP
==========================================

This script performs external validation using models trained on TCGA 
and evaluates them on PAIP dataset.

Models supported: Logistic Regression, KNN, Protonet, ANN, Random Forest

Configuration:
    Modify the variables in the CONFIGURATION section below.
"""

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from typing import Dict, List, Tuple, Optional

warnings.filterwarnings('ignore')


In [2]:
# ============================================================
# CONFIGURATION SECTION - ITERATE OVER ALL COMBINATIONS
# ============================================================

# --- BASE PATH CONFIGURATIONS ---
BASE_ROOT = r"D:\Aamir Gulzar\KSA_project2"
PROJECT_ROOT = os.path.join(BASE_ROOT, "Cancer-detection-classifier", "slide_classification")

# Ground truth labels
GROUND_TRUTH_PATH = os.path.join(BASE_ROOT, "paip_data", "labels", "paip_kfolds_71.csv")

# Output location
OUTPUT_BASE_PATH = os.path.join(PROJECT_ROOT, "TCGA_PAIP_EV_Results")
os.makedirs(OUTPUT_BASE_PATH, exist_ok=True)

# --- OPTIONS TO ITERATE OVER ---
# Define all combinations to evaluate
AGGREGATION_METHODS = ["PRISM"]#[
#     "Caption_based_aggregation", 
#     "Tissue_Type_Clustering", 
#     "Averaging", 
#     "TITAN"
# ]

MODEL_NAMES = ["PRISM"] #[
#     "H-Optimus-1", 
#     "Conch1_5", 
#     "UNI2", 
#     "Virchow2", 
#     "ConchV1"
# ]

TASKS = [
    "1-MSIH"
]

# Optionally, define a subset of combinations to evaluate
# Comment out above and use specific combinations to save time
# Example: run only a few key combinations
# """
# AGGREGATION_METHODS = ["TITAN", "Averaging"]
# MODEL_NAMES = ["Conch1_5", "UNI2"]
# TASKS = ["1-MSIH", "2-BRAF"]
# """

print("="*70)
print("EXTERNAL VALIDATION - TCGA TO PAIP (BATCH MODE)")
print("="*70)
print(f"Aggregation Methods: {len(AGGREGATION_METHODS)}")
print(f"  {', '.join(AGGREGATION_METHODS)}")
print(f"Model Names: {len(MODEL_NAMES)}")
print(f"  {', '.join(MODEL_NAMES)}")
print(f"Tasks: {len(TASKS)}")
print(f"  {', '.join(TASKS)}")
print(f"Total Combinations: {len(AGGREGATION_METHODS) * len(MODEL_NAMES) * len(TASKS)}")
print("="*70)


EXTERNAL VALIDATION - TCGA TO PAIP (BATCH MODE)
Aggregation Methods: 1
  PRISM
Model Names: 1
  PRISM
Tasks: 1
  1-MSIH
Total Combinations: 1


In [3]:
# ============================================================
# MODEL CONFIGURATIONS & HELPER FUNCTIONS
# ============================================================

# Define Base Dimensions for each model
MODEL_BASE_DIMS = {
    "Conch1_5": 768,
    "H-Optimus-1": 1536,
    "UNI2": 1536,
    "Virchow2": 2560,
    "ConchV1": 512,
    "PRISM": 1280
}

# Aggregation method multipliers
AGG_MULTIPLIERS = {
    "Caption_based_aggregation": 14,
    "Tissue_Type_Clustering": 9,
    "Averaging": 1,
    "TITAN": 1,
}

# PAIP feature folder mapping
PAIP_FEATURE_DIRS = {
    'Conch1_5': 'Conch1_5',
    'H-Optimus-1': 'H-Optimus-1', 
    'UNI2': 'UNI2',
    'Virchow2': 'Virchow2',
    'ConchV1': 'ConchV1',
    'PRISM':''
}

# All folds to evaluate
FOLDS = [0, 1, 2, 3]

# Classifier filename templates
CLASSIFIER_TEMPLATES = {
    'logistic_regression': 'fold{fold}_logistic_regression.pkl',
    'knn': 'fold{fold}_knn_model.pkl',
    'protonet': 'fold{fold}_protonet_model.pkl',
    'random_forest': 'fold{fold}_random_forest.pkl',
    'ann': 'fold{fold}_trained_ann_model_{input_dim}.pth',
}

# Classifiers to evaluate (comment out any you don't want to run)
CLASSIFIERS_TO_RUN = [
    'logistic_regression',
    'knn',
    'protonet',
    'random_forest',
    'ann'
]

# Helper function to calculate input dimension
def calculate_input_dim(model_name: str, aggregation_method: str) -> int:
    """Calculate input dimension based on model and aggregation method."""
    if model_name == "TITAN" or aggregation_method == "TITAN":
        return 768
    elif model_name == "PRISM" or aggregation_method == "PRISM":
        return 1280
    else:
        base = MODEL_BASE_DIMS.get(model_name, 768)
        mult = AGG_MULTIPLIERS.get(aggregation_method, 1)
        return int(base * mult)

# Helper function to get PAIP data path
def get_paip_data_path(base_root: str, aggregation_method: str, model_name: str) -> str:
    """Get the PAIP data directory for a specific aggregation method and model."""
    if aggregation_method == "PRISM":
        return os.path.join(base_root, "paip_data", "slide_aggregation", "PRISM")
    return os.path.join(base_root, "paip_data", "slide_aggregation", aggregation_method)


In [4]:

# ============================================================
# ANN MODEL ARCHITECTURE
# ============================================================
class ANNClassifier(nn.Module):
    """Must match the architecture used during TCGA training."""
    def __init__(self, input_dim: int, hidden_dim: int = 512,
                 hidden_dim2: int = 256, num_classes: int = 2):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim2, num_classes),
            nn.Softmax(dim=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)
    
# Model-specific architecture configurations - MATCH TCGA TRAINING
MODEL_ARCHITECTURES = {
    'Conch1_5': {'hidden_dim': 256, 'hidden_dim2': 128},
    'H-Optimus-1': {'hidden_dim': 256, 'hidden_dim2': 128},
    'UNI2': {'hidden_dim': 256, 'hidden_dim2': 128},
    'Virchow2': {'hidden_dim': 256, 'hidden_dim2': 128},
    'ConchV1': {'hidden_dim': 256, 'hidden_dim2': 128},
    'PRISM': {'hidden_dim': 256, 'hidden_dim2': 128}
}




In [5]:


# ============================================================
# LABEL & GROUND TRUTH HELPERS
# ============================================================
def parse_label(raw) -> Optional[int]:
    """
    Convert any label string/int to binary.
      nonMSIH / 0 / mss  →  0
      MSIH    / 1 / msi  →  1
    """
    if raw is None:
        return None
    s = str(raw).strip().lower()
    if s in ('0', 'nonmsih', 'non-msih', 'mss'):
        return 0
    if s in ('1', 'msih', 'msi-h', 'msih_c', 'msi'):
        return 1
    try:
        return int(float(s))
    except ValueError:
        return None


def load_ground_truth(csv_path: str) -> pd.DataFrame:
    """Load CSV and add normalised _wsi_id and _bin_label columns."""
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    print(f"Ground truth: {len(df)} rows | columns: {df.columns.tolist()}")
    print(df.head(3).to_string(), "\n")

    # Identify WSI id column
    id_col = 'WSI_Id' if 'WSI_Id' in df.columns else df.columns[0]

    # Identify label column
    lbl_col = None
    for cand in ('label', 'label_desc', 'Label', 'MSI_status'):
        if cand in df.columns:
            lbl_col = cand
            break
    if lbl_col is None:
        raise ValueError(f"No label column found in {df.columns.tolist()}")

    df['_wsi_id'] = df[id_col].astype(str).str.strip()
    df['_bin_label'] = df[lbl_col].apply(parse_label)

    bad = df['_bin_label'].isna().sum()
    if bad:
        print(f"⚠️  {bad} rows with unparseable labels — will be skipped")

    return df


def get_label(wsi_id: str, gt_df: pd.DataFrame) -> Optional[int]:
    """
    Look up binary label for a WSI id with three fallback strategies:
      1. Exact match on _wsi_id
      2. Partial / substring match
      3. Extract MSIH / nonMSIH directly from the filename stem
    """
    # Strategy 1 — exact
    row = gt_df[gt_df['_wsi_id'] == wsi_id]
    if not row.empty:
        return row['_bin_label'].values[0]

    # Strategy 2 — substring
    for _, r in gt_df.iterrows():
        if r['_wsi_id'] in wsi_id or wsi_id in r['_wsi_id']:
            return r['_bin_label']

    # Strategy 3 — parse from filename
    lower = wsi_id.lower()
    if '_nonmsih' in lower:
        return 0
    if '_msih' in lower:
        return 1

    return None

In [6]:


# ============================================================
# FEATURE LOADING & MODEL LOADING HELPERS
# ============================================================
def load_features(feature_dir: str) -> Tuple[Dict[str, torch.Tensor], List[str]]:
    """Load all .pt files from a directory. Flattens multi-dim tensors."""
    features_dict: Dict[str, torch.Tensor] = {}
    wsi_ids: List[str] = []

    if not os.path.isdir(feature_dir):
        print(f"  ❌  Directory not found: {feature_dir}")
        return features_dict, wsi_ids

    files = [f for f in os.listdir(feature_dir) if f.endswith('.pt')]
    print(f"  Found {len(files)} .pt files in: {os.path.basename(feature_dir)}")

    for fname in files:
        wsi_id = os.path.splitext(fname)[0]
        try:
            feat = torch.load(os.path.join(feature_dir, fname), map_location='cpu')
            if feat.dim() > 1:
                feat = feat.flatten()   # (N_crops, D) → (N_crops * D,)
            features_dict[wsi_id] = feat.float()
            wsi_ids.append(wsi_id)
        except Exception as e:
            print(f"  ⚠️  Could not load {fname}: {e}")

    return features_dict, wsi_ids


def load_model(classifier_name: str, model_path: str, input_dim: int, model_name: str = None):
    """Load either a sklearn (.pkl) or PyTorch ANN (.pth) model."""
    if classifier_name == 'ann':
        # Get model-specific architecture
        arch_config = MODEL_ARCHITECTURES.get(model_name, {'hidden_dim': 512, 'hidden_dim2': 256})
        m = ANNClassifier(input_dim, 
                         hidden_dim=arch_config['hidden_dim'],
                         hidden_dim2=arch_config['hidden_dim2'])
                         
        state = torch.load(model_path, map_location='cpu')
        # Handle both raw state_dict and wrapped checkpoints
        if isinstance(state, dict) and 'model_state_dict' in state:
            state = state['model_state_dict']
        m.model.load_state_dict(state)
        m.eval()
        return m
    else:
        loaded = joblib.load(model_path)
        # Handle protonet/random_forest checkpoint dict format
        if isinstance(loaded, dict):
            if 'model' in loaded:
                return loaded['model']
            elif 'classifier' in loaded:
                return loaded['classifier']
            elif 'prototypes' in loaded and 'labels_proto' in loaded:
                # This is a protonet model saved as dict - return the dict itself
                # We'll handle this special case in the predict function
                return loaded
        return loaded


def predict(classifier_name: str, model, feat: torch.Tensor):
    """
    Run inference for a single sample.
    Returns (pred: int, proba: np.ndarray shape (2,) or None)
    """
    if classifier_name == 'ann':
        x = feat.unsqueeze(0) if feat.dim() == 1 else feat
        with torch.no_grad():
            out = model(x)
            # Model already has Softmax, so out is already probabilities
            proba = out.numpy()[0]
            pred = int(out.argmax(dim=1).item())
        return pred, proba
    elif classifier_name == 'protonet' and isinstance(model, dict):
        # Handle protonet models saved as dicts with prototypes
        if 'prototypes' in model and 'labels_proto' in model:
            prototypes = model['prototypes']  # Shape: (n_prototypes, feature_dim)
            labels_proto = model['labels_proto']  # Shape: (n_prototypes,)
            
            # Convert to numpy if needed
            if isinstance(prototypes, torch.Tensor):
                prototypes = prototypes.numpy()
            if isinstance(labels_proto, torch.Tensor):
                labels_proto = labels_proto.numpy()
            
            # Compute distances to all prototypes
            feat_np = feat.numpy().reshape(1, -1)
            distances = np.linalg.norm(feat_np - prototypes, axis=1)
            
            # Find closest prototype
            closest_idx = np.argmin(distances)
            pred = int(labels_proto[closest_idx])
            
            # Compute probabilities based on distances (inverse distance weighting)
            inv_distances = 1.0 / (distances + 1e-8)  # Add small epsilon to avoid division by zero
            proba_raw = np.zeros(2)
            for i, label in enumerate(labels_proto):
                proba_raw[int(label)] += inv_distances[i]
            
            # Normalize to get probabilities
            proba = proba_raw / proba_raw.sum() if proba_raw.sum() > 0 else np.array([0.5, 0.5])
            
            return pred, proba
        else:
            raise ValueError(f"Protonet model dict missing required keys: {model.keys()}")
    else:
        X = feat.numpy().reshape(1, -1)
        pred = int(model.predict(X)[0])
        try:
            proba = model.predict_proba(X)[0]
        except Exception:
            proba = None
        return pred, proba



In [7]:

# ============================================================
# METRICS HELPER
# ============================================================
def compute_metrics(y_true: np.ndarray,
                    y_pred: np.ndarray,
                    y_proba: Optional[np.ndarray]) -> Dict:
    """
    Compute full classification metrics including:
    Accuracy, Balanced Accuracy, F1, AUROC,
    Sensitivity, Specificity, PPV, NPV, TP/TN/FP/FN
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0

    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'npv': npv,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'confusion_matrix': cm.tolist(),
        'auroc': None,
    }

    if y_proba is not None and len(np.unique(y_true)) == 2:
        try:
            pos = y_proba[:, 1] if y_proba.ndim == 2 else y_proba
            metrics['auroc'] = roc_auc_score(y_true, pos)
        except Exception:
            pass

    return metrics

In [8]:
# ============================================================
# MAIN INFERENCE FUNCTION (WITH PARAMETERS)
# ============================================================
def run_inference(model_name: str, aggregation_method: str, task: str, 
                 base_root: str, project_root: str, ground_truth_path: str,
                 output_base_path: str) -> Tuple[pd.DataFrame, int, int]:
    """
    Run inference for a specific model, aggregation method, and task combination.
    Returns (results_df, num_results, num_skipped)
    """
    
    # Calculate paths for this combination
    # tcga_models_base = os.path.join(project_root, "TCGA_Results", aggregation_method, model_name, task, "models")

    if aggregation_method == "PRISM":
        tcga_models_base = os.path.join(project_root, "TCGA_Results", "PRISM", task, "models")
    else:
        tcga_models_base = os.path.join(project_root, "TCGA_Results", aggregation_method, model_name, task, "models")
    paip_data_base = get_paip_data_path(base_root, aggregation_method, model_name)
    
    print(f"\n{'='*70}")
    print(f"CONFIGURATION: {aggregation_method} | {model_name} | {task}")
    print(f"{'='*70}")
    print(f"TCGA Models: {tcga_models_base}")
    print(f"PAIP Data: {paip_data_base}")
    
    # Load ground truth
    try:
        gt_df = load_ground_truth(ground_truth_path)
    except Exception as e:
        print(f"❌ Failed to load ground truth: {e}")
        return pd.DataFrame(), 0, 1

    all_results = []
    skipped_list = []

    # Load PAIP features
    paip_subdir = PAIP_FEATURE_DIRS.get(model_name, model_name)
    feat_dir = os.path.join(paip_data_base, paip_subdir) if paip_subdir else paip_data_base

    print(f"\nLoading PAIP features from: {feat_dir}")
    features_dict, wsi_ids = load_features(feat_dir)

    if not features_dict:
        print(f"  ❌  No features found — skipping this combination")
        return pd.DataFrame(), 0, 1

    input_dim = next(iter(features_dict.values())).shape[0]
    print(f"  ✓ Loaded {len(features_dict)} samples | Input dim: {input_dim}")

    # Build label map
    wsi_labels = {}
    missing_lbls = []
    for wsi_id in wsi_ids:
        lbl = get_label(wsi_id, gt_df)
        if lbl is not None:
            wsi_labels[wsi_id] = lbl
        else:
            missing_lbls.append(wsi_id)

    print(f"  ✓ Valid WSIs: {len(wsi_labels)}")
    if missing_lbls:
        print(f"  ⚠️  {len(missing_lbls)} WSIs without labels (will be skipped)")

    # Iterate through folds and classifiers
    for fold in FOLDS:
        for clf_name in CLASSIFIERS_TO_RUN:
            clf_template = CLASSIFIER_TEMPLATES[clf_name]
            model_filename = clf_template.format(fold=fold, input_dim=input_dim)
            model_path = os.path.join(tcga_models_base, model_filename)

            if not os.path.exists(model_path):
                skipped_list.append(model_path)
                continue

            # Load model
            try:
                model = load_model(clf_name, model_path, input_dim, model_name=model_name)
            except Exception as e:
                skipped_list.append(model_path)
                continue

            # Run predictions
            y_true_list = []
            y_pred_list = []
            y_proba_list = []
            wsi_id_list = []

            for wsi_id, feat in features_dict.items():
                if wsi_id not in wsi_labels:
                    continue

                try:
                    pred, proba = predict(clf_name, model, feat)
                    y_true_list.append(wsi_labels[wsi_id])
                    y_pred_list.append(pred)
                    y_proba_list.append(proba)
                    wsi_id_list.append(wsi_id)
                except Exception as e:
                    continue

            if len(y_true_list) == 0:
                continue

            # Compute metrics
            y_true = np.array(y_true_list)
            y_pred = np.array(y_pred_list)
            y_proba = np.array(y_proba_list) if y_proba_list[0] is not None else None

            metrics = compute_metrics(y_true, y_pred, y_proba)

            # Store results
            for i, wsi_id in enumerate(wsi_id_list):
                result_row = {
                    'fold': fold,
                    'classifier': clf_name,
                    'wsi_id': wsi_id,
                    'true_label': y_true[i],
                    'pred_label': y_pred[i],
                    'proba_class0': y_proba[i, 0] if y_proba is not None else None,
                    'proba_class1': y_proba[i, 1] if y_proba is not None else None,
                }
                result_row.update({f'metric_{k}': v for k, v in metrics.items()})
                all_results.append(result_row)

    # Format and save results
    output_dir = os.path.join(output_base_path, aggregation_method, model_name, task)
    os.makedirs(output_dir, exist_ok=True)
    
    if not all_results:
        print(f"❌ No results for this combination")
        return pd.DataFrame(), 0, len(skipped_list)
    
    df_detailed = pd.DataFrame(all_results)
    
    # Create individual predictions (simplified version)
    individual_predictions = []
    wsi_groups = df_detailed.groupby(['wsi_id', 'fold'])
    
    for (wsi_id, fold), group in wsi_groups:
        true_label = group['true_label'].iloc[0]
        
        pred_row = {
            'Fold': fold + 1,
            'WSI_ID': wsi_id,
            'Target': true_label,
            'lin_Pred': None, 'lin_Prob': None,
            'ann_Pred': None, 'ann_Prob': None,
            'knn_Pred': None, 'knn_Prob': None,
            'proto_Pred': None, 'proto_Prob': None,
            'rf_Pred': None, 'rf_Prob': None
        }
        
        classifier_mapping = {
            'logistic_regression': 'lin',
            'ann': 'ann',
            'knn': 'knn',
            'protonet': 'proto',
            'random_forest': 'rf'
        }
        
        for _, row in group.iterrows():
            classifier = row['classifier']
            if classifier in classifier_mapping:
                prefix = classifier_mapping[classifier]
                pred_row[f'{prefix}_Pred'] = row['pred_label']
                if row['proba_class1'] is not None:
                    pred_row[f'{prefix}_Prob'] = round(row['proba_class1'], 4)
        
        individual_predictions.append(pred_row)
    
    # Create summary metrics
    summary_metrics = []
    classifiers = df_detailed['classifier'].unique()
    folds = sorted(df_detailed['fold'].unique())
    
    classifier_order = ['logistic_regression', 'ann', 'knn', 'protonet', 'random_forest']
    classifier_mapping = {
        'logistic_regression': 'lin',
        'ann': 'ann', 
        'knn': 'knn',
        'protonet': 'proto',
        'random_forest': 'rf'
    }
    
    for classifier in classifier_order:
        if classifier not in classifiers:
            continue
            
        classifier_data = df_detailed[df_detailed['classifier'] == classifier]
        prefix = classifier_mapping[classifier]
        
        metric_types = [
            ('acc', 'metric_accuracy'),
            ('bacc', 'metric_balanced_accuracy'), 
            ('macro_f1', 'metric_f1_macro'),
            ('weighted_f1', 'metric_f1_weighted'),
            ('auroc', 'metric_auroc'),
            ('conf_matrix', 'metric_confusion_matrix')
        ]
        
        for metric_suffix, metric_key in metric_types:
            metric_row = {'Metric': f'{prefix}_{metric_suffix}'}
            
            fold_values = []
            for fold in folds:
                fold_data = classifier_data[classifier_data['fold'] == fold]
                if not fold_data.empty:
                    value = fold_data[metric_key].iloc[0]
                    if metric_suffix == 'conf_matrix':
                        metric_row[f'Fold{fold+1}'] = str(value)
                    else:
                        if value is not None:
                            rounded_value = round(float(value), 10)
                            metric_row[f'Fold{fold+1}'] = rounded_value
                            fold_values.append(rounded_value)
                        else:
                            metric_row[f'Fold{fold+1}'] = None
                else:
                    metric_row[f'Fold{fold+1}'] = None
            
            if metric_suffix != 'conf_matrix' and fold_values:
                metric_row['AvgFolds'] = round(sum(fold_values) / len(fold_values), 10)
            else:
                metric_row['AvgFolds'] = None if metric_suffix != 'conf_matrix' else ''
                
            summary_metrics.append(metric_row)
    
    # Convert to DataFrames
    df_individual = pd.DataFrame(individual_predictions)
    df_summary = pd.DataFrame(summary_metrics)
    
    if not df_individual.empty:
        df_individual = df_individual.sort_values(['Fold', 'WSI_ID']).reset_index(drop=True)
        
        required_columns = [
            'Fold', 'WSI_ID', 'Target',
            'lin_Pred', 'lin_Prob',
            'ann_Pred', 'ann_Prob', 
            'knn_Pred', 'knn_Prob',
            'proto_Pred', 'proto_Prob',
            'rf_Pred', 'rf_Prob'
        ]
        
        existing_cols = [col for col in required_columns if col in df_individual.columns]
        df_individual = df_individual[existing_cols]
    
    if not df_summary.empty:
        summary_columns = ['Metric', 'Fold1', 'Fold2', 'Fold3', 'Fold4', 'AvgFolds']
        existing_summary_cols = [col for col in summary_columns if col in df_summary.columns]
        df_summary = df_summary[existing_summary_cols]
    
    # Save to CSV
    individual_file = os.path.join(output_dir, "external_validation_PAIP_individual.csv")
    summary_file = os.path.join(output_dir, "external_validation_PAIP_summary.csv")
    
    if not df_individual.empty:
        df_individual.to_csv(individual_file, index=False)
    
    if not df_summary.empty:
        df_summary.to_csv(summary_file, index=False)

    print(f"✓ Results saved → {output_dir}")
    
    return df_individual, len(individual_predictions), len(skipped_list)


In [9]:
# ============================================================
# BATCH PROCESSING: ITERATE OVER ALL COMBINATIONS
# ============================================================
def run_batch_validation():
    """
    Run validation for all combinations of aggregation methods,
    model names, and tasks.
    """
    all_results_summary = {
        'completed': [],
        'failed': [],
        'skipped': []
    }
    
    total_combinations = len(AGGREGATION_METHODS) * len(MODEL_NAMES) * len(TASKS)
    processed_count = 0
    
    print("\n" + "="*70)
    print(f"BATCH PROCESSING: {total_combinations} combinations")
    print("="*70)
    
    for agg_method in AGGREGATION_METHODS:
        for model_name in MODEL_NAMES:
            for task in TASKS:
                processed_count += 1
                combo_id = f"{agg_method} | {model_name} | {task}"
                print(f"\n[{processed_count}/{total_combinations}] {combo_id}")
                
                try:
                    results_df, num_results, num_skipped = run_inference(
                        model_name=model_name,
                        aggregation_method=agg_method,
                        task=task,
                        base_root=BASE_ROOT,
                        project_root=PROJECT_ROOT,
                        ground_truth_path=GROUND_TRUTH_PATH,
                        output_base_path=OUTPUT_BASE_PATH
                    )
                    
                    if num_results > 0:
                        all_results_summary['completed'].append({
                            'combo': combo_id,
                            'predictions': num_results,
                            'skipped_models': num_skipped
                        })
                    else:
                        all_results_summary['skipped'].append(combo_id)
                        
                except Exception as e:
                    print(f"❌ Error: {e}")
                    all_results_summary['failed'].append({
                        'combo': combo_id,
                        'error': str(e)
                    })
    
    # Print summary
    print("\n" + "="*70)
    print("BATCH PROCESSING SUMMARY")
    print("="*70)
    print(f"✓ Completed: {len(all_results_summary['completed'])}")
    for item in all_results_summary['completed']:
        print(f"  • {item['combo']}: {item['predictions']} predictions")
    
    print(f"\n⊘ Skipped: {len(all_results_summary['skipped'])}")
    for item in all_results_summary['skipped']:
        print(f"  • {item}")
    
    print(f"\n❌ Failed: {len(all_results_summary['failed'])}")
    for item in all_results_summary['failed']:
        print(f"  • {item['combo']}: {item['error']}")
    
    print(f"\nTotal: {processed_count}/{total_combinations}")
    print(f"Output Directory: {OUTPUT_BASE_PATH}")
    print("="*70)
    
    return all_results_summary


# ============================================================
# RUN BATCH VALIDATION
# ============================================================
if __name__ == "__main__":
    batch_summary = run_batch_validation()



BATCH PROCESSING: 1 combinations

[1/1] PRISM | PRISM | 1-MSIH

CONFIGURATION: PRISM | PRISM | 1-MSIH
TCGA Models: D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\TCGA_Results\PRISM\1-MSIH\models
PAIP Data: D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\PRISM
Ground truth: 78 rows | columns: ['WSI_Id', 'label']
                     WSI_Id    label
0     training_data_01_MSIH     MSIH
1  training_data_02_nonMSIH  nonMSIH
2  training_data_03_nonMSIH  nonMSIH 


Loading PAIP features from: D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\PRISM
  Found 73 .pt files in: PRISM
  ✓ Loaded 73 samples | Input dim: 1280
  ✓ Valid WSIs: 73
✓ Results saved → D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\TCGA_PAIP_EV_Results\PRISM\PRISM\1-MSIH

BATCH PROCESSING SUMMARY
✓ Completed: 1
  • PRISM | PRISM | 1-MSIH: 292 predictions

⊘ Skipped: 0

❌ Failed: 0

Total: 1/1
Output Directory: D:\Aamir Gulzar\KSA_project2\Cancer-

In [10]:
# ============================================================
# UTILITIES FOR REVIEWING RESULTS
# ============================================================

def review_results_summary(output_base_path: str):
    """
    Review and summarize all results from batch processing.
    Shows which combinations have results and their basic stats.
    """
    print("\n" + "="*70)
    print("RESULTS REVIEW")
    print("="*70)
    
    for agg_method in AGGREGATION_METHODS:
        for model_name in MODEL_NAMES:
            for task in TASKS:
                result_dir = os.path.join(output_base_path, agg_method, model_name, task)
                individual_file = os.path.join(result_dir, "external_validation_PAIP_individual.csv")
                
                if os.path.exists(individual_file):
                    try:
                        df = pd.read_csv(individual_file)
                        print(f"\n✓ {agg_method} | {model_name} | {task}")
                        print(f"  Samples: {len(df)} | Folds: {sorted(df['Fold'].unique())}")
                    except Exception as e:
                        print(f"⚠️  {agg_method} | {model_name} | {task}: {e}")

def create_per_sample_predictions_df(results_data: List[Dict]) -> pd.DataFrame:
    """
    Create per-sample predictions DataFrame with EXACT format requirements.
    
    Args:
        results_data: List of dictionaries containing prediction results
        
    Returns:
        DataFrame with columns: Fold, WSI_ID, Target, lin_Pred, lin_Prob, 
        ann_Pred, ann_Prob, knn_Pred, knn_Prob, proto_Pred, proto_Prob, rf_Pred, rf_Prob
    """
    # Convert to DataFrame
    df = pd.DataFrame(results_data)
    
    # Ensure 1-indexed folds
    if 'Fold' in df.columns:
        df['Fold'] = df['Fold'].apply(lambda x: x + 1 if x < 1 else x)
    
    # Define EXACT column order
    required_columns = [
        'Fold', 'WSI_ID', 'Target',
        'lin_Pred', 'lin_Prob',
        'ann_Pred', 'ann_Prob', 
        'knn_Pred', 'knn_Prob',
        'proto_Pred', 'proto_Prob',
        'rf_Pred', 'rf_Prob'
    ]
    
    # Add missing columns with None values
    for col in required_columns:
        if col not in df.columns:
            df[col] = None
    
    # Reorder columns to match exact requirements
    df = df[required_columns]
    
    # CRITICAL: Sort by Fold first, then WSI_ID
    # This ensures all Fold 1 samples are together, then all Fold 2, etc.
    df = df.sort_values(['Fold', 'WSI_ID']).reset_index(drop=True)
    
    return df


def create_summary_metrics_df(metrics_data: List[Dict]) -> pd.DataFrame:
    """
    Create summary metrics DataFrame with EXACT format requirements.
    
    Args:
        metrics_data: List of dictionaries containing metric results
        
    Returns:
        DataFrame with columns: Metric, Fold1, Fold2, Fold3, Fold4, AvgFolds
    """
    df = pd.DataFrame(metrics_data)
    
    # Define EXACT column order
    required_columns = ['Metric', 'Fold1', 'Fold2', 'Fold3', 'Fold4', 'AvgFolds']
    
    # Add missing columns with None values
    for col in required_columns:
        if col not in df.columns:
            df[col] = None
    
    # Reorder columns to match exact requirements
    existing_cols = [col for col in required_columns if col in df.columns]
    df = df[existing_cols]
    
    return df

def save_dataframes(df_individual: pd.DataFrame, 
                   df_summary: pd.DataFrame,
                   individual_path: str,
                   summary_path: str):
    """
    Save DataFrames to CSV files.
    
    Args:
        df_individual: Per-sample predictions DataFrame
        df_summary: Summary metrics DataFrame  
        individual_path: Path to save individual predictions CSV
        summary_path: Path to save summary metrics CSV
    """
    # Save individual predictions
    if not df_individual.empty:
        df_individual.to_csv(individual_path, index=False)
    
    # Save summary metrics
    if not df_summary.empty:
        df_summary.to_csv(summary_path, index=False)


In [11]:
# ============================================================
# CONFIGURATION VERIFICATION
# ============================================================

def verify_configuration():
    """Verify that all combinations are configured correctly."""
    print("\n" + "="*70)
    print("CONFIGURATION VERIFICATION")
    print("="*70)
    
    print(f"\n✓ AGGREGATION METHODS ({len(AGGREGATION_METHODS)}):")
    for i, agg in enumerate(AGGREGATION_METHODS, 1):
        print(f"  {i}. {agg}")
    
    print(f"\n✓ MODEL NAMES ({len(MODEL_NAMES)}):")
    for i, model in enumerate(MODEL_NAMES, 1):
        print(f"  {i}. {model}")
    
    print(f"\n✓ TASKS ({len(TASKS)}):")
    for i, task in enumerate(TASKS, 1):
        print(f"  {i}. {task}")
    
    total = len(AGGREGATION_METHODS) * len(MODEL_NAMES) * len(TASKS)
    print(f"\n✓ TOTAL COMBINATIONS: {total}")
    print(f"  {len(AGGREGATION_METHODS)} aggregation × {len(MODEL_NAMES)} models × {len(TASKS)} tasks = {total}")
    
    print("\n" + "="*70)
    print("✓ Configuration is ready for batch processing")
    print("="*70)

# Run verification
verify_configuration()



CONFIGURATION VERIFICATION

✓ AGGREGATION METHODS (1):
  1. PRISM

✓ MODEL NAMES (1):
  1. PRISM

✓ TASKS (1):
  1. 1-MSIH

✓ TOTAL COMBINATIONS: 1
  1 aggregation × 1 models × 1 tasks = 1

✓ Configuration is ready for batch processing
